In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import sqlite3

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")

print("All imports successful")

All imports successful


In [13]:
DB_PATH = Path("data/db/bluestock_mf.db")
REPORTS_PATH = Path("reports")
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB_PATH)

query_nav = """
SELECT f.scheme_name, n.date, n.nav
FROM fact_nav n
JOIN dim_fund f ON n.amfi_code = f.amfi_code
WHERE f.category = 'Equity'
ORDER BY n.date
"""
nav_df = pd.read_sql(query_nav, conn)

fig1 = go.Figure()
for scheme in nav_df['scheme_name'].unique()[:10]:
    data = nav_df[nav_df['scheme_name'] == scheme]
    fig1.add_trace(go.Scatter(x=data['date'], y=data['nav'], mode='lines', name=scheme[:25]))

fig1.add_vrect(x0="2023-03-01", x1="2023-12-01", fillcolor="green", opacity=0.1, line_width=0)
fig1.add_vrect(x0="2024-08-01", x1="2024-11-01", fillcolor="red", opacity=0.1, line_width=0)
fig1.update_layout(title="NAV Trends - Top 10 Equity Funds (2022-2026)", xaxis_title="Date", yaxis_title="NAV (Rupees)", height=600)
fig1.write_html(REPORTS_PATH / "nav_trend_plotly.html")
fig1.write_image(REPORTS_PATH / "nav_trend_top10.png")

print("Saved: reports/nav_trend_plotly.html, reports/nav_trend_top10.png")

Saved: reports/nav_trend_plotly.html, reports/nav_trend_top10.png


In [14]:
query_aum = """
SELECT strftime('%Y', date) as year, fund_house, aum_lakh_crore
FROM fact_aum
WHERE date >= '2022-01-01'
"""
aum_df = pd.read_sql(query_aum, conn)

plt.figure(figsize=(14, 7))
sns.barplot(data=aum_df, x='fund_house', y='aum_lakh_crore', hue='year')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Fund House')
plt.ylabel('AUM (Lakh Crore)')
plt.title('AUM Growth by Fund House (2022-2025)')
plt.legend(title='Year')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'aum_growth_barchart.png', dpi=150)
plt.close()

sbi_2025 = aum_df[(aum_df['fund_house'] == 'SBI Mutual Fund') & (aum_df['year'] == '2025')]['aum_lakh_crore'].values
if len(sbi_2025) > 0:
    print(f"SBI Mutual Fund AUM (Dec 2025): Rs.{sbi_2025[0]} Lakh Crore")
print("Saved: reports/aum_growth_barchart.png")

SBI Mutual Fund AUM (Dec 2025): Rs.12.5 Lakh Crore
Saved: reports/aum_growth_barchart.png


In [15]:
from pathlib import Path
RAW_PATH = Path("data/raw")

sip_df = pd.read_csv(RAW_PATH / "04_monthly_sip_inflows.csv")
sip_df['month'] = pd.to_datetime(sip_df['month'])

fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=sip_df['month'], y=sip_df['sip_inflow_crore'], mode='lines+markers', name='SIP Inflow', line=dict(width=2)))

fig3.add_annotation(x="2025-12-01", y=31002, text="All-time High: Rs.31,002 Cr (Dec 2025)", showarrow=True, arrowhead=2, ax=50, ay=-30)
fig3.add_hline(y=31002, line_dash="dash", line_color="red")

fig3.update_layout(title="Monthly SIP Inflow Trends (Jan 2022 - Dec 2025)", xaxis_title="Month", yaxis_title="SIP Inflow (Rs. Crore)", height=500)
fig3.write_html(REPORTS_PATH / "sip_inflow_plotly.html")
fig3.write_image(REPORTS_PATH / "sip_inflow_trend.png")

print("Saved: reports/sip_inflow_plotly.html, reports/sip_inflow_trend.png")

Saved: reports/sip_inflow_plotly.html, reports/sip_inflow_trend.png


In [16]:
cat_df = pd.read_csv(RAW_PATH / "05_category_inflows.csv")
cat_pivot = cat_df.pivot(index='category', columns='month', values='net_inflow_crore')

plt.figure(figsize=(14, 8))
sns.heatmap(cat_pivot, cmap='RdYlGn', center=0, annot=True, fmt='.0f', cbar_kws={'label': 'Net Inflow (Rs. Crore)'})
plt.title('Category-wise Monthly Net Inflows (FY 2024-25)')
plt.xlabel('Month')
plt.ylabel('Fund Category')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'category_inflow_heatmap.png', dpi=150)
plt.close()

print("Saved: reports/category_inflow_heatmap.png")

Saved: reports/category_inflow_heatmap.png


In [17]:
PROCESSED_PATH = Path("data/processed")
trans_df = pd.read_csv(PROCESSED_PATH / "08_investor_transactions_clean.csv")

fig5, axes = plt.subplots(2, 2, figsize=(12, 10))

age_counts = trans_df['age_group'].value_counts()
axes[0, 0].pie(age_counts.values, labels=age_counts.index, autopct='%1.1f%%', startangle=90)
axes[0, 0].set_title('Investor Distribution by Age Group')

sip_data = trans_df[trans_df['transaction_type'] == 'SIP']
sip_by_age = [sip_data[sip_data['age_group'] == age]['amount_inr'] for age in age_counts.index]
axes[0, 1].boxplot(sip_by_age, tick_labels=age_counts.index)
axes[0, 1].set_xlabel('Age Group')
axes[0, 1].set_ylabel('SIP Amount (Rs.)')
axes[0, 1].set_title('SIP Amount Distribution by Age Group')

gender_counts = trans_df['gender'].value_counts()
axes[1, 0].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%', startangle=90)
axes[1, 0].set_title('Gender Distribution')

gender_sip = sip_data.groupby('gender')['amount_inr'].mean()
axes[1, 1].bar(gender_sip.index, gender_sip.values, color=['blue', 'pink'])
axes[1, 1].set_xlabel('Gender')
axes[1, 1].set_ylabel('Avg SIP Amount (Rs.)')
axes[1, 1].set_title('Average SIP Amount by Gender')

plt.tight_layout()
plt.savefig(REPORTS_PATH / 'investor_demographics.png', dpi=150)
plt.close()

print("Saved: reports/investor_demographics.png")

Saved: reports/investor_demographics.png


In [18]:
fig6, axes = plt.subplots(1, 2, figsize=(14, 6))

state_sip = sip_data.groupby('state')['amount_inr'].sum().sort_values(ascending=False).head(10)
axes[0].barh(state_sip.index, state_sip.values, color='teal')
axes[0].set_xlabel('Total SIP Amount (Rs.)')
axes[0].set_title('Top 10 States by SIP Investment')

tier_split = trans_df.groupby('city_tier')['amount_inr'].sum()
axes[1].pie(tier_split.values, labels=tier_split.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Investment Split: T30 vs B30 Cities')

plt.tight_layout()
plt.savefig(REPORTS_PATH / 'geo_distribution.png', dpi=150)
plt.close()

print("Saved: reports/geo_distribution.png")

Saved: reports/geo_distribution.png


In [19]:
folio_df = pd.read_csv(RAW_PATH / "06_industry_folio_count.csv")
folio_df['month'] = pd.to_datetime(folio_df['month'])
folio_df = folio_df.sort_values('month')

fig7 = go.Figure()
fig7.add_trace(go.Scatter(x=folio_df['month'], y=folio_df['total_folios_crore'], mode='lines+markers', name='Total Folios', line=dict(width=2, color='green')))

fig7.update_layout(title="Mutual Fund Folio Count Growth (Jan 2022 - Dec 2025)", xaxis_title="Month", yaxis_title="Total Folios (Crore)", height=500)
fig7.write_html(REPORTS_PATH / "folio_growth_plotly.html")
fig7.write_image(REPORTS_PATH / "folio_growth.png")

print("Saved: reports/folio_growth_plotly.html, reports/folio_growth.png")

Saved: reports/folio_growth_plotly.html, reports/folio_growth.png


In [20]:
query_corr = """
SELECT n.amfi_code, f.scheme_name, n.date, n.daily_return
FROM fact_nav n
JOIN dim_fund f ON n.amfi_code = f.amfi_code
WHERE f.category = 'Equity'
AND n.daily_return IS NOT NULL
"""
corr_df = pd.read_sql(query_corr, conn)

top_funds = corr_df.groupby('scheme_name')['daily_return'].count().nlargest(10).index
corr_df_filtered = corr_df[corr_df['scheme_name'].isin(top_funds)]
pivot_corr = corr_df_filtered.pivot(index='date', columns='scheme_name', values='daily_return')
corr_matrix = pivot_corr.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('NAV Return Correlation Matrix - Top 10 Equity Funds')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'correlation_matrix.png', dpi=150)
plt.close()

print("Saved: reports/correlation_matrix.png")

Saved: reports/correlation_matrix.png


In [21]:
holdings_df = pd.read_csv(RAW_PATH / "09_portfolio_holdings.csv")
sector_allocation = holdings_df.groupby('sector')['market_value_cr'].sum().sort_values(ascending=False).head(8)

plt.figure(figsize=(8, 8))
plt.pie(sector_allocation.values, labels=sector_allocation.index, autopct='%1.1f%%', startangle=90, pctdistance=0.85)
centre_circle = plt.Circle((0, 0), 0.70, fc='white')
plt.gca().add_artist(centre_circle)
plt.title('Top 8 Sector Allocations - Equity Mutual Funds')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'sector_allocation_donut.png', dpi=150)
plt.close()

print("Saved: reports/sector_allocation_donut.png")

Saved: reports/sector_allocation_donut.png
